# 13. 최종 Tableau용 주제분류 테이블 생성

12번 노트북의 ML 자동분류 + GPT mini fallback 결과를 최종 라벨로 확정하고, 원본 `intellytics_display_online_voc` row에 `memo_id`와 최종 주제를 붙인 Tableau 연동용 테이블을 생성합니다.

- 입력: `ml_classification_detail`
- 최종 detail: `classification_detail_final`
- Tableau용 원본 row 확장 테이블: `classification_tableau_final`


In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.final_classification_builder as final_classification_builder

importlib.reload(config_loader)
importlib.reload(final_classification_builder)

from common.config_loader import load_config, get_output_table, get_reference_table, get_source_table
from ml.final_classification_builder import (
    build_and_save_final_classification_outputs,
    summarize_ml_classification_result,
)

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

print("settings =", config["path"]["settings"])
print("source =", get_source_table(config, "raw_review_table"))
print("ml_detail =", get_output_table(config, "ml_classification_detail"))
print("final_detail =", get_output_table(config, "classification_detail_final"))
print("tableau_final =", get_output_table(config, "classification_tableau_final"))


In [ ]:
# 1. 12번 결과 검증: unresolved_pending_fallback_rows가 0에 가까울수록 최종화 준비가 잘 된 상태입니다.
stage12_summary = summarize_ml_classification_result(spark, config)
stage12_summary


In [ ]:
# 2-3. 최종 detail 생성 + 원본 row에 memo_id/topic을 붙인 Tableau용 테이블 저장
# 같은 prompt_version/taxonomy_version 결과는 replace_version 방식으로 중복 없이 재생성합니다.
result = build_and_save_final_classification_outputs(
    spark,
    config,
    write_mode="replace_version",
)

result


In [ ]:
# 최종 Tableau 테이블 row 수와 topic 분포 확인
tableau_table = get_output_table(config, "classification_tableau_final")
category_mapping_table = get_reference_table(config, "category_mapping_table")

tableau_df = spark.table(tableau_table).where(F.col("prompt_version") == config["version"]["prompt_version"])

display(
    tableau_df.alias("t")
    .join(
        spark.table(category_mapping_table).alias("m"),
        on=["cate_1_depth", "cate_2_depth"],
        how="left",
    )
    .groupBy(
        "t.cate_1_depth",
        "m.cate_1_depth_kor",
        "t.cate_2_depth",
        "m.cate_2_depth_kor",
        "t.sc_measurement",
        "t.pred_topic_type",
        "t.pred_topic",
    )
    .agg(
        F.count("*").alias("raw_row_cnt"),
        F.countDistinct("memo_id").alias("distinct_memo_id_cnt"),
        F.avg("confidence_score").alias("avg_confidence"),
    )
    .orderBy("t.cate_1_depth", "t.cate_2_depth", "t.sc_measurement", F.desc("raw_row_cnt"))
)


In [ ]:
# Tableau 연결 전 샘플 확인
display(
    tableau_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "year",
        "country",
        "brand_name",
        "device_type",
        "memo_id",
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "confidence_score",
        "llm_used_yn",
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    .limit(100)
)
